# microWakeWord v2 — High-Accuracy `"Hey Kiwi"` Training (Alexa-Parity)

This Colab notebook trains a quantized streaming INT8 `.tflite` wake-word model for **`"Hey Kiwi"`** engineered to match the accuracy, low false-trigger rate, and ESP32-S3 TFLite Micro compatibility of the official **`alexa.tflite`** model.

## Why the previous `"kiwi"` run failed (and what is fixed here)
1. **Language & Speaker Diversity (`es_ES` 2 voices → English 1,020+ speakers):** The previous run accidentally used Spanish (`es_ES`) with only 2 speakers (`1,000` clips total). This notebook uses English (`en_US` + `en_GB` + South-Asian-friendly voices): `en_US-libritts_r-medium` (**904 speakers**), `en_GB-vctk-medium` (**109 speakers**, including diverse UK/Indian/international accents), plus **10 individual voices** (`en_US-kusal-medium`, `lessac`, `amy`, `ryan`, `hfc_female`, `hfc_male`, `cori`, `alan`, `alba`, `northern_english_male`) to synthesize **5,000+ diverse positives**.
2. **AudioSet 404 Fixed:** `agkphysics/AudioSet` (`bal_train09.tar`) returned `404`, leaving `audioset_16k` empty. Cell 8 now streams real AudioSet clips via HuggingFace `datasets` (`balanced` split, `Audio(decode=False)`) + `soundfile` + `soxr`, with an automatic `ESC-50` fallback.
3. **Targeted Hard Negatives (Partial + Phonetic + Hinglish):** `3,500+` hard negatives across **64 confusable phrases** — protecting against partial wake words (`"hey"`, `"kiwi"`, `"a kiwi"`, `"hey there"`), rival assistants (`"hey siri"`, `"hey alexa"`, `"hey google"`, `"hey jarvis"`), phonetic rhymes (`"hey kiki"`, `"hey kevin"`, `"hey key"`, `"hey we"`, `"key we"`), and Indian/Hinglish phrases (`"hey kavi"`, `"hey ravi"`, `"hey kiran"`, `"hey bhai"`, `"hey yaar"`).
4. **Ambient False-Positive Evaluation (`dinner_party_eval`) Free-Colab Safe:** Fixes `sampling_weight: 0.0` (was `1.0`), runs Piper/AudioSet/mmap extraction in isolated subprocesses, splits training vs quantized TFLite streaming eval into two separate processes so training RAM is freed before export, and on Free Colab (`<= 16.5 GB` RAM) uses a read-only in-memory clip cap on the unmodified `dinner_party_eval` (`>= 35` min/split of whole unmodified clips, zero disk rewriting) so peak RAM stays safely under ~12.7 GB while minimizing `ambient_false_positives_per_hour`.
5. **Exact `alexa.tflite` Architecture & `WakeWordEngine.cpp` Compatibility:** Uses `mixednet` with `--mixconv_kernel_sizes '[5], [9], [13], [21]'`, `--pointwise_filters '64,64,64,64'`, `--first_conv_filters 32`, `--first_conv_kernel_size 5`, `--stride 3`, `--repeat_in_block '1, 1, 1, 1'`, `--residual_connection '0,0,0,0'` — producing **only the exact 12 TFLite Micro ops** registered in `WakeWordEngine.cpp` (no `SPLIT`/`ADD` ops, compact ~20 KB tensor arena).
6. **Streaming Verification, Cutoff Calibration & Bundle Export:** Cell 12 runs exact `WakeWordEngine.cpp`-style streaming inference (`pymicro_features` + `INT8` scaling + `stride=3` + `sliding_window_size=5`), verifies ops compatibility, sweeps cutoffs, and exports `hey_kiwi.tflite`, `hey_kiwi.json`, and `hey_kiwi_model.h`.

## Before you start
1. In Colab: **Runtime → Change runtime type → T4 GPU**.
2. Then click **Runtime → Run all** (no blocking `input()` prompts!). Total runtime is ~45–60 minutes on a free T4 GPU.

In [ ]:
# =============================================================================
# Cell 1: CONFIGURATION — Review/edit before running
# =============================================================================
# IMPORTANT: Runtime -> Change runtime type -> T4 GPU (before running any cell)
# =============================================================================

import json

CONFIG = {
    # Output naming
    'WAKE_WORD_NAME': 'hey_kiwi',
    'WAKE_WORD_DISPLAY': 'Hey Kiwi',

    # Timing & inference profile (matches alexa.json / WakeWordEngine.cpp)
    # 1500ms matches alexa.tflite (both "alexa" and "hey kiwi" are 3 syllables)
    # and keeps frame counts cleanly compatible with stride=3 and <20 KB arena.
    'CLIP_DURATION_MS': 1500,
    'SLIDING_WINDOW_SIZE': 5,
    'PROBABILITY_CUTOFF': 0.90,            # Alexa default; calibrated in Cell 12

    # Natural English intonation/pause variations for "hey kiwi"
    # (Avoid hyphenated phonetic respellings like "kee-wee" which espeak mispronounces as "kay-way")
    'TEST_VARIANTS': [
        'hey kiwi',
        'hey, kiwi',
        'hey kiwi!',
        'hey kiwi?',
    ],
    'SELECTED_VARIANT_INDICES': [0, 1, 2, 3],

    # Synthetic sample counts (generated across 1,020+ multi-speaker English voices)
    'TARGET_POSITIVE_SAMPLES': 5000,
    'TARGET_HARD_NEG_SAMPLES': 3500,
    'POSITIVE_SAMPLING_WEIGHT': 5.0,

    # Curated Piper English multi-speaker + single-speaker voices (US + UK + South-Asian-friendly)
    'PIPER_VOICES': [
        'en_US-libritts_r-medium',         # 904 speakers (US English)
        'en_GB-vctk-medium',               # 109 speakers (UK/Indian/international accents)
        'en_US-kusal-medium',              # South-Asian-origin US English voice
        'en_US-lessac-medium',
        'en_US-amy-medium',
        'en_US-ryan-medium',
        'en_US-hfc_female-medium',
        'en_US-hfc_male-medium',
        'en_GB-cori-medium',
        'en_GB-alan-medium',
        'en_GB-alba-medium',
        'en_GB-northern_english_male-medium',
    ],

    # Hard negatives: partial "hey" phrases, rival wake words, phonetic lookalikes, Hinglish words
    # (Do NOT put "kiwi" alone in here, or the streaming CNN penalizes the word "kiwi" itself!)
    'HARD_NEGATIVE_WORDS': [
        # Partial "hey" protection (so "hey ..." without "kiwi" never triggers)
        'hey', 'hey there', 'hey you', 'hey guy', 'hey guys', 'hey buddy',
        'hey man', 'hey bro', 'hey friend', 'hey everyone',
        'hey look', 'hey listen', 'hey wait', 'hey what',
        # Competing assistants
        'hey siri', 'hey google', 'hey alexa', 'hey jarvis',
        'hey cortana', 'hey bixby', 'hey computer',
        # Phonetic lookalikes of "hey kiwi"
        'hey key', 'hey we', 'hey tv', 'hey kiki', 'hey kevin',
        'hey kelly', 'hey kenny', 'hey katie', 'hey kitty', 'hey kylie',
        'hey mickey', 'hey ricky', 'hey vicky', 'hey nicky', 'hey stevie',
        'hey sleepy', 'hey easy', 'hey really', 'hey give me', 'hey leave me',
        'hey see me', 'hey keep it', 'hey creepy', 'hey cheeky',
        'key we', 'take it easy', 'a key', 'heavy',
        # Indian / Hinglish lookalikes & common conversational words
        'hey kavi', 'hey ravi', 'hey kiran', 'hey kabir',
        'hey kishan', 'hey krish', 'hey bhai', 'hey yaar',
        'hey kya', 'okay',
    ],
    'HARD_NEG_SAMPLING_WEIGHT': 2.0,

    # Augmentation dataset cap (streamed cleanly from agkphysics/AudioSet)
    'AUDIOSET_MAX_CLIPS': 2500,

    # Two-stage training schedule (balanced ratio so positives reach 0.85-0.99 confidence)
    'TRAINING_STEPS': 16000,
    'TRAINING_STEPS_STAGE2': 6000,
    'LEARNING_RATE_STAGE1': 0.001,
    'LEARNING_RATE_STAGE2': 0.0001,
    'NEGATIVE_CLASS_WEIGHT_STAGE1': 8,
    'NEGATIVE_CLASS_WEIGHT_STAGE2': 12,
    'EVAL_STEP_INTERVAL': 2000,
    'BATCH_SIZE': 128,

    # Ambient false-positive checkpoint selection (REQUIRED for Alexa-grade zero-FP accuracy)
    # 'auto' uses a 25-min/split in-memory cap (~15,000 eval windows) on Free Colab (<=16.5GB RAM)
    # and full dinner_party_eval on High-RAM runtimes.
    'USE_DINNER_PARTY_EVAL': True,
    'DINNER_PARTY_EVAL_MODE': 'auto',      # 'auto', 'lite', or 'full'
    'DINNER_PARTY_LITE_MINUTES': 25,       # caps 119-min CHiME6 clip in-memory (~450MB vs 4.3GB!)
    'TARGET_FALSE_ACCEPTS_PER_HOUR': 0.5,

    # Optional features
    'ADD_CUSTOM_RECORDINGS': False,        # True = upload your own voice .wav/.mp3/.m4a in Cell 7
    'CLEAN_TRAIN_DIR': True,               # True = fresh run (set False only when resuming mid-training)
    'SAVE_TO_GOOGLE_DRIVE': False,         # True = copy final export bundle to Google Drive
}

# Preserve VOICE_INFO/TOTAL_SPEAKERS if Cell 4 already populated /content/config.json
import os
if os.path.exists('/content/config.json'):
    try:
        with open('/content/config.json') as f:
            _prev = json.load(f)
        for _k in ('VOICE_INFO', 'TOTAL_SPEAKERS'):
            if _k in _prev:
                CONFIG[_k] = _prev[_k]
    except Exception:
        pass

indices = CONFIG.get('SELECTED_VARIANT_INDICES', list(range(len(CONFIG['TEST_VARIANTS']))))
CONFIG['VARIANTS'] = [CONFIG['TEST_VARIANTS'][i] for i in indices]

with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"Wake word          : {CONFIG['WAKE_WORD_DISPLAY']} ({CONFIG['WAKE_WORD_NAME']})")
print(f"Clip / Window      : {CONFIG['CLIP_DURATION_MS']}ms / sliding_window={CONFIG['SLIDING_WINDOW_SIZE']} / cutoff={CONFIG['PROBABILITY_CUTOFF']}")
print(f"Positive target    : {CONFIG['TARGET_POSITIVE_SAMPLES']} clips across {len(CONFIG['PIPER_VOICES'])} voice packs")
print(f"Hard negatives     : {CONFIG['TARGET_HARD_NEG_SAMPLES']} clips across {len(CONFIG['HARD_NEGATIVE_WORDS'])} phrases")
print(f"Training schedule  : {CONFIG['TRAINING_STEPS']} + {CONFIG['TRAINING_STEPS_STAGE2']} steps (neg weights {CONFIG['NEGATIVE_CLASS_WEIGHT_STAGE1']} -> {CONFIG['NEGATIVE_CLASS_WEIGHT_STAGE2']})")
print(f"Dinner party eval  : {'ON (target <= ' + str(CONFIG['TARGET_FALSE_ACCEPTS_PER_HOUR']) + ' FP/hr)' if CONFIG['USE_DINNER_PARTY_EVAL'] else 'OFF'}")
print("Saved /content/config.json")

In [ ]:
# =============================================================================
# Cell 2: Check GPU & Install Dependencies
# =============================================================================

import subprocess, sys, importlib, site

# 1. Fail fast if GPU runtime is not enabled
gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError(
        'No GPU detected! In Colab go to: Runtime -> Change runtime type -> T4 GPU, then re-run.'
    )
print(gpu_check.stdout.splitlines()[0:3])

# 2. Install exact compatible versions for Colab Python 3.12/3.13
%pip install -q 'datasets==2.21.0'
%pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
!test -d /content/microWakeWord || git clone -q https://github.com/OHF-Voice/micro-wake-word /content/microWakeWord
%pip install -q -e /content/microWakeWord
%pip install -q piper-tts soundfile pydub soxr scipy psutil

importlib.invalidate_caches()
site.main()

import datasets
import microwakeword
import piper
import pymicro_features
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'Python           : {sys.version.split()[0]}')
print(f'datasets         : {datasets.__version__}')
print(f'tensorflow       : {tf.__version__} (GPUs: {len(gpus)})')
print('microwakeword    : OK')
print('piper-tts        : OK')
print('pymicro_features : OK')

if datasets.__version__ != '2.21.0':
    print('\n*** RESTART REQUIRED: older datasets version was already cached in RAM. ***')
    print('*** Click: Runtime -> Restart session, then run Cell 1 and continue from Cell 3. ***')
else:
    print('\nReady. Continue with Cell 3.')

In [ ]:
# =============================================================================
# Cell 3: Patch microWakeWord Stride Assertion & Verify Runtime
# =============================================================================

import importlib, re

import datasets
importlib.reload(datasets)
assert datasets.__version__ == '2.21.0', (
    f'datasets {datasets.__version__} loaded, need 2.21.0. '
    'Run Runtime -> Restart session, then run Cell 1 and continue from Cell 3.'
)

utils_path = '/content/microWakeWord/microwakeword/utils.py'
with open(utils_path, 'r') as f:
    code = f.read()

pattern = re.compile(r'^([ \t]*)assert\s+spectrogram\.shape\[0\]\s*%\s*stride\s*==\s*0.*$', re.MULTILINE)
match = pattern.search(code)
if match:
    indent = match.group(1)
    replacement = (
        f'{indent}remainder = spectrogram.shape[0] % stride\n'
        f'{indent}if remainder != 0:\n'
        f'{indent}    spectrogram = spectrogram[:spectrogram.shape[0] - remainder]'
    )
    code = pattern.sub(replacement, code, count=1)
    with open(utils_path, 'w') as f:
        f.write(code)
    print('Patched microwakeword/utils.py stride truncation cleanly.')
else:
    print('microwakeword/utils.py already patched (or stride assertion not present).')

In [ ]:
# =============================================================================
# Cell 4: Download Curated Multi-Speaker & Multi-Accent English Piper Voices
# =============================================================================

import json, os, time, urllib.request

with open('/content/config.json') as f:
    CONFIG = json.load(f)

voices_dir = '/content/piper_voices'
os.makedirs(voices_dir, exist_ok=True)
voices_json_path = '/content/voices.json'

def download_with_retry(url, dest, expected_bytes=None, retries=3):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        if expected_bytes is None or os.path.getsize(dest) == expected_bytes:
            return
    for attempt in range(1, retries + 1):
        try:
            tmp = dest + '.tmp'
            urllib.request.urlretrieve(url, tmp)
            if os.path.getsize(tmp) == 0:
                raise RuntimeError('Downloaded empty file')
            if expected_bytes is not None and os.path.getsize(tmp) != expected_bytes:
                raise RuntimeError(
                    f'Size mismatch: got {os.path.getsize(tmp)}, expected {expected_bytes}'
                )
            os.replace(tmp, dest)
            return
        except Exception as e:
            if attempt == retries:
                raise RuntimeError(f'Failed downloading {url}: {e}')
            time.sleep(2 * attempt)

download_with_retry(
    'https://huggingface.co/rhasspy/piper-voices/resolve/main/voices.json',
    voices_json_path,
)

with open(voices_json_path) as f:
    all_voices = json.load(f)

HF_ROOT = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/'
downloaded_voices = {}
total_speakers = 0

for key in CONFIG['PIPER_VOICES']:
    if key not in all_voices:
        print(f'Skipping missing voice key: {key}')
        continue
    info = all_voices[key]
    files = info.get('files', {})
    onnx_rel = next((p for p in files if p.endswith('.onnx')), None)
    json_rel = next((p for p in files if p.endswith('.onnx.json')), None)
    if not onnx_rel or not json_rel:
        print(f'Skipping incomplete voice entry: {key}')
        continue

    onnx_dest = os.path.join(voices_dir, f'{key}.onnx')
    json_dest = os.path.join(voices_dir, f'{key}.onnx.json')
    download_with_retry(HF_ROOT + onnx_rel, onnx_dest, files[onnx_rel].get('size_bytes'))
    download_with_retry(HF_ROOT + json_rel, json_dest, files[json_rel].get('size_bytes'))

    with open(json_dest) as vf:
        vmeta = json.load(vf)
    num_spk = max(1, int(vmeta.get('num_speakers', info.get('num_speakers', 1))))
    total_speakers += num_spk
    downloaded_voices[key] = {
        'onnx': onnx_dest,
        'json': json_dest,
        'num_speakers': num_spk,
        'sample_rate': int(vmeta.get('audio', {}).get('sample_rate', 22050)),
    }
    print(f'  Ready: {key:36s} | speakers={num_spk}')

assert len(downloaded_voices) > 0, 'No Piper voices downloaded!'
CONFIG['VOICE_INFO'] = downloaded_voices
CONFIG['TOTAL_SPEAKERS'] = total_speakers

with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f'\nTotal voices ready: {len(downloaded_voices)} packs, {total_speakers} unique speakers.')

In [ ]:
# =============================================================================
# Cell 5: Preview Pronunciation Variants (Non-Blocking for Run-All)
# =============================================================================

import inspect, io, json, os, wave
import numpy as np
from IPython.display import Audio, HTML, display
from piper.voice import PiperVoice

try:
    from piper.voice import SynthesisConfig
except ImportError:
    from piper import SynthesisConfig

with open('/content/config.json') as f:
    CONFIG = json.load(f)

def make_syn_config(speaker_id=None, length_scale=1.0, noise_scale=0.667, noise_w=0.8):
    sig = inspect.signature(SynthesisConfig)
    kwargs = {}
    if 'speaker_id' in sig.parameters and speaker_id is not None:
        kwargs['speaker_id'] = int(speaker_id)
    if 'length_scale' in sig.parameters:
        kwargs['length_scale'] = float(length_scale)
    if 'noise_scale' in sig.parameters:
        kwargs['noise_scale'] = float(noise_scale)
    if 'noise_w_scale' in sig.parameters:
        kwargs['noise_w_scale'] = float(noise_w)
    elif 'noise_w' in sig.parameters:
        kwargs['noise_w'] = float(noise_w)
    if 'normalize_audio' in sig.parameters:
        kwargs['normalize_audio'] = True
    return SynthesisConfig(**kwargs)

def synth_preview_wav(voice, text, out_path, speaker_id=None):
    cfg = make_syn_config(speaker_id=speaker_id)
    if hasattr(voice, 'synthesize_wav'):
        sig = inspect.signature(voice.synthesize_wav)
        kw = {}
        if 'syn_config' in sig.parameters:
            kw['syn_config'] = cfg
        elif 'config' in sig.parameters:
            kw['config'] = cfg
        with wave.open(out_path, 'wb') as wf:
            voice.synthesize_wav(text, wf, **kw)
    else:
        with wave.open(out_path, 'wb') as wf:
            voice.synthesize(text, wf, speaker_id=speaker_id)

os.makedirs('/content/preview_wavs', exist_ok=True)
preview_voice_key = (
    'en_US-kusal-medium'
    if 'en_US-kusal-medium' in CONFIG['VOICE_INFO']
    else list(CONFIG['VOICE_INFO'].keys())[0]
)
voice = PiperVoice.load(CONFIG['VOICE_INFO'][preview_voice_key]['onnx'])

for idx, phrase in enumerate(CONFIG['TEST_VARIANTS']):
    wav_path = f'/content/preview_wavs/variant_{idx}.wav'
    synth_preview_wav(voice, phrase, wav_path)
    display(HTML(f'<b>Variant #{idx}:</b> <code>{phrase}</code> ({preview_voice_key})'))
    display(Audio(wav_path))

# Also preview a multi-speaker UK voice ("en_GB-vctk-medium") for accent check
if 'en_GB-vctk-medium' in CONFIG['VOICE_INFO']:
    uk_voice = PiperVoice.load(CONFIG['VOICE_INFO']['en_GB-vctk-medium']['onnx'])
    uk_path = '/content/preview_wavs/uk_vctk_sample.wav'
    synth_preview_wav(uk_voice, CONFIG['TEST_VARIANTS'][0], uk_path, speaker_id=0)
    display(HTML('<b>Accent check (en_GB-vctk-medium spk 0):</b>'))
    display(Audio(uk_path))
    del uk_voice

del voice

indices = CONFIG.get('SELECTED_VARIANT_INDICES', list(range(len(CONFIG['TEST_VARIANTS']))))
CONFIG['VARIANTS'] = [CONFIG['TEST_VARIANTS'][i] for i in indices]

with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"Active training variants: {CONFIG['VARIANTS']}")

In [ ]:
# =============================================================================
# Cell 6: Generate Multi-Speaker Positives & Hard Negatives (Subprocess)
# =============================================================================
# Runs inside an isolated subprocess so ONNX Runtime / Piper memory is 100% freed afterward.

script = r'''
import inspect, io, json, os, random, shutil, time, wave
import numpy as np
import scipy.io.wavfile
import soxr
from piper.voice import PiperVoice

try:
    from piper.voice import SynthesisConfig
except ImportError:
    from piper import SynthesisConfig

TARGET_SR = 16000
RNG = random.Random(42)

with open('/content/config.json') as f:
    CONFIG = json.load(f)

VOICE_INFO = CONFIG.get('VOICE_INFO')
if not VOICE_INFO:
    # Auto-discover cached Piper voices if Cell 1 was re-run after Cell 4
    VOICE_INFO = {}
    voices_dir = '/content/piper_voices'
    if os.path.isdir(voices_dir):
        for fn in sorted(os.listdir(voices_dir)):
            if fn.endswith('.onnx'):
                key = fn[:-5]
                onnx_p = os.path.join(voices_dir, fn)
                json_p = onnx_p + '.json'
                if os.path.exists(json_p):
                    with open(json_p) as vf:
                        vmeta = json.load(vf)
                    VOICE_INFO[key] = {
                        'onnx': onnx_p,
                        'json': json_p,
                        'num_speakers': max(1, int(vmeta.get('num_speakers', 1))),
                        'sample_rate': int(vmeta.get('audio', {}).get('sample_rate', 22050)),
                    }
    assert VOICE_INFO, 'No voices found! Run Cell 4 first to download Piper voices.'

VARIANTS = CONFIG.get('VARIANTS') or [
    CONFIG['TEST_VARIANTS'][i]
    for i in CONFIG.get('SELECTED_VARIANT_INDICES', list(range(len(CONFIG['TEST_VARIANTS']))))
]
HARD_WORDS = CONFIG['HARD_NEGATIVE_WORDS']

SYN_PARAMS = set(inspect.signature(SynthesisConfig).parameters.keys())

def build_syn_config(speaker_id, length_scale, noise_scale, noise_w):
    kw = {}
    if 'speaker_id' in SYN_PARAMS and speaker_id is not None:
        kw['speaker_id'] = int(speaker_id)
    if 'length_scale' in SYN_PARAMS:
        kw['length_scale'] = float(length_scale)
    if 'noise_scale' in SYN_PARAMS:
        kw['noise_scale'] = float(noise_scale)
    if 'noise_w_scale' in SYN_PARAMS:
        kw['noise_w_scale'] = float(noise_w)
    elif 'noise_w' in SYN_PARAMS:
        kw['noise_w'] = float(noise_w)
    if 'normalize_audio' in SYN_PARAMS:
        kw['normalize_audio'] = True
    return SynthesisConfig(**kw)

def synth_16k_trimmed(voice, text, speaker_id, length_scale, noise_scale, noise_w):
    buf = io.BytesIO()
    cfg = build_syn_config(speaker_id, length_scale, noise_scale, noise_w)
    if hasattr(voice, 'synthesize_wav'):
        sig = inspect.signature(voice.synthesize_wav)
        kw = {}
        if 'syn_config' in sig.parameters:
            kw['syn_config'] = cfg
        elif 'config' in sig.parameters:
            kw['config'] = cfg
        with wave.open(buf, 'wb') as wf:
            voice.synthesize_wav(text, wf, **kw)
    else:
        with wave.open(buf, 'wb') as wf:
            voice.synthesize(text, wf, speaker_id=speaker_id)

    buf.seek(0)
    with wave.open(buf, 'rb') as wf:
        sr = wf.getframerate()
        pcm_bytes = wf.readframes(wf.getnframes())

    audio = np.frombuffer(pcm_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    if audio.size == 0:
        return None
    if sr != TARGET_SR:
        audio = soxr.resample(audio, sr, TARGET_SR)

    pcm = np.clip(np.round(audio * 32767.0), -32768, 32767).astype(np.int16)

    # Gentle silence trim preserving 60ms onset/offset margin so "hey" is never clipped
    active = np.flatnonzero(np.abs(pcm) > 120)
    if active.size == 0:
        return None
    pad = int(0.060 * TARGET_SR)
    start = max(0, int(active[0]) - pad)
    end = min(pcm.shape[0], int(active[-1]) + pad)
    pcm = pcm[start:end]

    if pcm.shape[0] < int(0.18 * TARGET_SR) or np.max(np.abs(pcm)) < 500:
        return None
    return pcm

def allocate_voice_quotas(total_count):
    # Balanced allocation: ~55% LibriTTS-R (904 spk), ~25% VCTK (109 spk), ~20% individual packs
    quotas = {}
    singles = [k for k, v in VOICE_INFO.items() if v['num_speakers'] == 1]
    has_libritts = 'en_US-libritts_r-medium' in VOICE_INFO
    has_vctk = 'en_GB-vctk-medium' in VOICE_INFO

    remaining = total_count
    if has_libritts:
        q = int(total_count * 0.55)
        quotas['en_US-libritts_r-medium'] = q
        remaining -= q
    if has_vctk:
        q = int(total_count * 0.25)
        quotas['en_GB-vctk-medium'] = q
        remaining -= q
    if singles:
        per_single, rem = divmod(remaining, len(singles))
        for idx, k in enumerate(singles):
            quotas[k] = per_single + (1 if idx < rem else 0)
    elif quotas:
        first_k = next(iter(quotas))
        quotas[first_k] += remaining
    return quotas

def generate_dataset(out_dir, phrases, total_count, label, max_dur_ms=None):
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir, exist_ok=True)
    quotas = allocate_voice_quotas(total_count)
    idx = 0
    durations_ms = []
    t0 = time.time()

    for vkey, count in quotas.items():
        if count <= 0:
            continue
        vinfo = VOICE_INFO[vkey]
        voice = PiperVoice.load(vinfo['onnx'])
        num_spk = vinfo['num_speakers']
        spk_order = list(range(num_spk))
        RNG.shuffle(spk_order)

        for i in range(count):
            spk = spk_order[i % num_spk] if num_spk > 1 else None
            if (i + 1) % num_spk == 0 and num_spk > 1:
                RNG.shuffle(spk_order)

            phrase = phrases[idx % len(phrases)]
            length_scale = RNG.uniform(0.80, 1.20)
            noise_scale = RNG.uniform(0.45, 0.85)
            noise_w = RNG.uniform(0.50, 0.95)

            pcm = synth_16k_trimmed(voice, phrase, spk, length_scale, noise_scale, noise_w)
            if pcm is None:
                pcm = synth_16k_trimmed(voice, phrases[0], spk, 0.95, 0.667, 0.8)
            if pcm is None:
                continue

            dur_ms = (pcm.shape[0] * 1000.0) / TARGET_SR
            if max_dur_ms is not None and dur_ms > max_dur_ms:
                # Re-synthesize slightly faster if unusually stretched
                pcm_fast = synth_16k_trimmed(voice, phrase, spk, 0.82, noise_scale, noise_w)
                if pcm_fast is not None and pcm_fast.shape[0] < pcm.shape[0]:
                    pcm = pcm_fast
                    dur_ms = (pcm.shape[0] * 1000.0) / TARGET_SR

            scipy.io.wavfile.write(os.path.join(out_dir, f'{idx}.wav'), TARGET_SR, pcm)
            durations_ms.append(dur_ms)
            idx += 1

            if idx % 500 == 0:
                elapsed = time.time() - t0
                rate = idx / max(elapsed, 1e-6)
                eta = (total_count - idx) / max(rate, 1e-6)
                print(f'[{label}] {idx}/{total_count} clips ({rate:.1f}/s, ETA {eta:.0f}s)')

        del voice

    arr = np.array(durations_ms, dtype=np.float32)
    print(
        f'[{label}] Done: {idx} WAVs in {time.time()-t0:.1f}s | '
        f'dur ms: min={arr.min():.0f}, p50={np.percentile(arr,50):.0f}, '
        f'p95={np.percentile(arr,95):.0f}, max={arr.max():.0f}'
    )

max_pos_ms = CONFIG['CLIP_DURATION_MS'] - 250
generate_dataset(
    '/content/generated_samples',
    VARIANTS,
    CONFIG['TARGET_POSITIVE_SAMPLES'],
    'POSITIVES',
    max_dur_ms=max_pos_ms,
)

# Shuffle hard negative phrases cleanly so every voice pack covers all 64 phrases
hard_phrases = list(HARD_WORDS)
RNG.shuffle(hard_phrases)
generate_dataset(
    '/content/hard_negatives',
    hard_phrases,
    CONFIG['TARGET_HARD_NEG_SAMPLES'],
    'HARD_NEG',
)
'''

with open('/content/generate_piper_samples.py', 'w') as f:
    f.write(script)

!python /content/generate_piper_samples.py

In [ ]:
# =============================================================================
# Cell 7: Optional — Add Your Own Mic Recordings of "Hey Kiwi"
# =============================================================================
# Controlled by CONFIG['ADD_CUSTOM_RECORDINGS'] (False by default so Run-All stays unattended).
# If True, upload 5-25 .wav/.mp3/.m4a/.ogg files of your voice saying "hey kiwi".

import json, os, shutil
import numpy as np
import scipy.io.wavfile

with open('/content/config.json') as f:
    CONFIG = json.load(f)

if not CONFIG.get('ADD_CUSTOM_RECORDINGS', False):
    print('ADD_CUSTOM_RECORDINGS is False — skipping manual upload.')
else:
    from google.colab import files
    from pydub import AudioSegment

    custom_dir = '/content/custom_user_recordings'
    shutil.rmtree(custom_dir, ignore_errors=True)
    os.makedirs(custom_dir, exist_ok=True)
    os.chdir(custom_dir)
    uploaded = files.upload()
    os.chdir('/content')

    existing = len([x for x in os.listdir('/content/generated_samples') if x.endswith('.wav')])
    added = 0
    for fname in uploaded.keys():
        src_path = os.path.join(custom_dir, fname)
        try:
            seg = AudioSegment.from_file(src_path).set_frame_rate(16000).set_channels(1).set_sample_width(2)
            pcm = np.array(seg.get_array_of_samples(), dtype=np.int16)
            active = np.flatnonzero(np.abs(pcm) > 180)
            if active.size > 0:
                pad = int(0.08 * 16000)
                pcm = pcm[max(0, int(active[0]) - pad):min(len(pcm), int(active[-1]) + pad)]
            # Replicate 15x with mild gain scaling so user's room mic & accent are strongly weighted
            for rep in range(15):
                gain = 0.70 + (rep % 7) * 0.10
                scaled = np.clip(np.round(pcm.astype(np.float32) * gain), -32768, 32767).astype(np.int16)
                out_name = f'/content/generated_samples/custom_{existing + added}.wav'
                scipy.io.wavfile.write(out_name, 16000, scaled)
                added += 1
        except Exception as e:
            print(f'Skipped {fname}: {e}')
    print(f'Added {added} augmented copies from {len(uploaded)} custom recordings.')

In [ ]:
# =============================================================================
# Cell 8: Download Augmentation Data (MIT RIRs, AudioSet, FMA) in Subprocess
# =============================================================================
# Fixes the broken bal_train09.tar 404 by streaming agkphysics/AudioSet (balanced)
# and runs in a subprocess so datasets/pydub/librosa RAM is freed before training.

aug_script = r'''
import io, json, os, shutil, urllib.request, zipfile
from pathlib import Path
import numpy as np
import scipy.io.wavfile
import soundfile as sf
import soxr
from datasets import Audio, load_dataset
from pydub import AudioSegment

TARGET_SR = 16000

with open('/content/config.json') as f:
    CONFIG = json.load(f)

def to_16k_int16(data, sr):
    if data.ndim > 1:
        data = data.mean(axis=1)
    data = data.astype(np.float32)
    if sr != TARGET_SR:
        data = soxr.resample(data, sr, TARGET_SR)
    return np.clip(np.round(data * 32767.0), -32768, 32767).astype(np.int16)

# 1. MIT Room Impulse Responses
rir_dir = '/content/mit_rirs'
if not os.path.exists(rir_dir) or len(os.listdir(rir_dir)) < 100:
    os.makedirs(rir_dir, exist_ok=True)
    print('[1/3] Downloading MIT RIRs...')
    rir_ds = load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train')
    rir_ds = rir_ds.cast_column('audio', Audio(decode=False))
    saved = 0
    for i, row in enumerate(rir_ds):
        try:
            a = row['audio']
            if a.get('bytes'):
                data, sr = sf.read(io.BytesIO(a['bytes']), dtype='float32')
            elif a.get('path') and os.path.exists(a['path']):
                data, sr = sf.read(a['path'], dtype='float32')
            else:
                continue
            pcm = to_16k_int16(data, sr)
            scipy.io.wavfile.write(os.path.join(rir_dir, f'rir_{i}.wav'), TARGET_SR, pcm)
            saved += 1
        except Exception:
            pass
    print(f'[1/3] MIT RIRs ready: {saved} WAVs')
else:
    print(f'[1/3] MIT RIRs cached: {len(os.listdir(rir_dir))} WAVs')

# 2. AudioSet (streamed balanced split -> 16kHz mono WAVs; fallback to ESC-50 if unreachable)
audioset_dir = '/content/audioset_16k'
max_as = int(CONFIG.get('AUDIOSET_MAX_CLIPS', 2500))
if not os.path.exists(audioset_dir) or len(os.listdir(audioset_dir)) < max_as:
    os.makedirs(audioset_dir, exist_ok=True)
    saved = len(os.listdir(audioset_dir))
    print(f'[2/3] Streaming AudioSet (balanced, target {max_as} clips)...')
    try:
        as_ds = None
        for cfg_args in [('balanced', False), ('balanced', True), (None, False)]:
            try:
                kw = {'split': 'train', 'streaming': True}
                if cfg_args[1]:
                    kw['trust_remote_code'] = True
                if cfg_args[0] is not None:
                    as_ds = load_dataset('agkphysics/AudioSet', cfg_args[0], **kw)
                else:
                    as_ds = load_dataset('agkphysics/AudioSet', **kw)
                break
            except Exception:
                as_ds = None
        if as_ds is None:
            raise RuntimeError('Could not stream agkphysics/AudioSet')
        as_ds = as_ds.cast_column('audio', Audio(decode=False))

        for i, row in enumerate(as_ds):
            if saved >= max_as:
                break
            try:
                a = row['audio']
                raw_bytes = a.get('bytes')
                if raw_bytes is None and a.get('path') and os.path.exists(a['path']):
                    with open(a['path'], 'rb') as bf:
                        raw_bytes = bf.read()
                if not raw_bytes:
                    continue
                try:
                    data, sr = sf.read(io.BytesIO(raw_bytes), dtype='float32')
                    pcm = to_16k_int16(data, sr)
                except Exception:
                    seg = AudioSegment.from_file(io.BytesIO(raw_bytes)).set_frame_rate(TARGET_SR).set_channels(1).set_sample_width(2)
                    pcm = np.array(seg.get_array_of_samples(), dtype=np.int16)
                if pcm.size < TARGET_SR:
                    continue
                scipy.io.wavfile.write(os.path.join(audioset_dir, f'as_{saved}.wav'), TARGET_SR, pcm)
                saved += 1
                if saved % 500 == 0:
                    print(f'      AudioSet: {saved}/{max_as} WAVs')
            except Exception:
                continue
    except Exception as e:
        print(f'      AudioSet stream error ({e}), falling back to ashraq/esc50...')
        esc_ds = load_dataset('ashraq/esc50', split='train', streaming=True)
        esc_ds = esc_ds.cast_column('audio', Audio(decode=False))
        for i, row in enumerate(esc_ds):
            if saved >= max_as:
                break
            try:
                a = row['audio']
                raw_bytes = a.get('bytes')
                if not raw_bytes:
                    continue
                data, sr = sf.read(io.BytesIO(raw_bytes), dtype='float32')
                pcm = to_16k_int16(data, sr)
                scipy.io.wavfile.write(os.path.join(audioset_dir, f'esc_{saved}.wav'), TARGET_SR, pcm)
                saved += 1
            except Exception:
                continue
    print(f'[2/3] AudioSet/Background ready: {len(os.listdir(audioset_dir))} WAVs')
else:
    print(f'[2/3] AudioSet cached: {len(os.listdir(audioset_dir))} WAVs')

# 3. FMA music background
fma_16k_dir = '/content/fma_16k'
if not os.path.exists(fma_16k_dir) or len(os.listdir(fma_16k_dir)) < 100:
    os.makedirs('/content/fma', exist_ok=True)
    os.makedirs(fma_16k_dir, exist_ok=True)
    zip_path = '/content/fma/fma_xs.zip'
    print('[3/3] Downloading FMA xsmall...')
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip',
        zip_path,
    )
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/fma')
    saved = 0
    for mp3 in Path('/content/fma').glob('**/*.mp3'):
        try:
            audio = AudioSegment.from_mp3(str(mp3)).set_frame_rate(TARGET_SR).set_channels(1).set_sample_width(2)
            samples = np.array(audio.get_array_of_samples(), dtype=np.int16)
            if samples.size >= TARGET_SR:
                scipy.io.wavfile.write(os.path.join(fma_16k_dir, mp3.stem + '.wav'), TARGET_SR, samples)
                saved += 1
        except Exception:
            pass
    shutil.rmtree('/content/fma', ignore_errors=True)
    print(f'[3/3] FMA ready: {saved} WAVs (raw zip cleaned)')
else:
    print(f'[3/3] FMA cached: {len(os.listdir(fma_16k_dir))} WAVs')
'''

with open('/content/prepare_aug_data.py', 'w') as f:
    f.write(aug_script)

!python /content/prepare_aug_data.py

In [ ]:
# =============================================================================
# Cell 9: Build Augmented Spectrogram Mmap Features (Subprocess)
# =============================================================================
# Runs in a subprocess so audiomentations/librosa/numba RAM (~2-3 GB) is freed before training.

feat_script = r'''
import os, shutil
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

bg_paths = [
    p for p in ['/content/fma_16k', '/content/audioset_16k']
    if os.path.isdir(p) and len(os.listdir(p)) > 0
]
rir_paths = [
    p for p in ['/content/mit_rirs']
    if os.path.isdir(p) and len(os.listdir(p)) > 0
]
assert len(bg_paths) == 2, f'Expected both fma_16k and audioset_16k, found: {bg_paths}'
assert len(rir_paths) == 1, f'Expected mit_rirs, found: {rir_paths}'
print(f'Background dirs: {bg_paths}')
print(f'RIR dirs       : {rir_paths}')

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15,
        'TanhDistortion': 0.15,
        'PitchShift': 0.20,
        'BandStopFilter': 0.10,
        'AddColorNoise': 0.15,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.50,
    },
    impulse_paths=rir_paths,
    background_paths=bg_paths,
    background_min_snr_db=-3,
    background_max_snr_db=15,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

def extract_mmaps(in_wav_dir, out_feat_dir, label):
    shutil.rmtree(out_feat_dir, ignore_errors=True)
    os.makedirs(out_feat_dir, exist_ok=True)
    clips = Clips(
        input_directory=in_wav_dir,
        file_pattern='*.wav',
        max_clip_duration_s=None,
        remove_silence=False,
        random_split_seed=10,
        split_count=0.1,
    )
    for split in ['training', 'validation', 'testing']:
        split_dir = os.path.join(out_feat_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        split_name = {'training': 'train', 'validation': 'validation', 'testing': 'test'}[split]
        repetition = 2 if split == 'training' else 1
        slide = 1 if split == 'testing' else 10

        specs = SpectrogramGeneration(
            clips=clips,
            augmenter=augmenter,
            slide_frames=slide,
            step_ms=10,
        )
        mmap_dir = os.path.join(split_dir, 'wakeword_mmap')
        RaggedMmap.from_generator(
            out_dir=mmap_dir,
            sample_generator=specs.spectrogram_generator(split=split_name, repeat=repetition),
            batch_size=100,
            verbose=True,
        )
        count = len(RaggedMmap(mmap_dir))
        print(f'  [{label}] {split:10s}: {count} spectrograms')

extract_mmaps('/content/generated_samples', '/content/generated_augmented_features', 'POSITIVES')
extract_mmaps('/content/hard_negatives', '/content/generated_hard_negative_features', 'HARD_NEG')
print('All spectrogram mmap features built cleanly.')
'''

with open('/content/build_features.py', 'w') as f:
    f.write(feat_script)

!python /content/build_features.py

In [ ]:
# =============================================================================
# Cell 10: Download Negative Datasets & Write training_parameters.yaml
# =============================================================================

import gc, json, os, shutil, subprocess, zipfile, psutil, yaml
from mmap_ninja.ragged import RaggedMmap

with open('/content/config.json') as f:
    CONFIG = json.load(f)

neg_dir = '/content/negative_datasets'
os.makedirs(neg_dir, exist_ok=True)

# Clean up any leftover dinner_party_eval_lite from older runs
shutil.rmtree(os.path.join(neg_dir, 'dinner_party_eval_lite'), ignore_errors=True)

link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
datasets_to_get = [
    ('dinner_party.zip', 'dinner_party'),
    ('dinner_party_eval.zip', 'dinner_party_eval'),
    ('no_speech.zip', 'no_speech'),
    ('speech.zip', 'speech'),
]

for fname, folder in datasets_to_get:
    target_folder = os.path.join(neg_dir, folder)
    if os.path.exists(target_folder) and len(os.listdir(target_folder)) > 0:
        print(f'Cached: {target_folder}')
        continue
    zip_path = os.path.join(neg_dir, fname)
    print(f'Downloading {fname}...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', zip_path, link_root + fname], check=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(neg_dir)
    os.remove(zip_path)   # free disk space immediately

# Verify unmodified dinner_party_eval integrity
dp_eval_dir = '/content/negative_datasets/dinner_party_eval'
for dirpath, _, filenames in os.walk(dp_eval_dir):
    if os.path.basename(dirpath) == 'wakeword_mmap':
        mm = RaggedMmap(dirpath)
        first_shape = mm[0].shape if len(mm) > 0 else None
        print(f'Verified {os.path.relpath(dirpath, dp_eval_dir)}: {len(mm)} clips (first shape={first_shape})')
        del mm

# Build training_parameters.yaml
if CONFIG.get('CLEAN_TRAIN_DIR', False):
    shutil.rmtree(f"/content/trained_models/{CONFIG['WAKE_WORD_NAME']}", ignore_errors=True)

STAGE1 = int(CONFIG['TRAINING_STEPS'])
STAGE2 = int(CONFIG.get('TRAINING_STEPS_STAGE2', 0))
NEG_W1 = int(CONFIG.get('NEGATIVE_CLASS_WEIGHT_STAGE1', 20))
NEG_W2 = int(CONFIG.get('NEGATIVE_CLASS_WEIGHT_STAGE2', 30))
LR1 = float(CONFIG.get('LEARNING_RATE_STAGE1', 0.001))
LR2 = float(CONFIG.get('LEARNING_RATE_STAGE2', 0.0001))

if STAGE2 > 0:
    steps = [STAGE1, STAGE2]
    lrs = [LR1, LR2]
    neg_weights = [NEG_W1, NEG_W2]
else:
    steps = [STAGE1]
    lrs = [LR1]
    neg_weights = [NEG_W1]
n = len(steps)

USE_DP = bool(CONFIG.get('USE_DINNER_PARTY_EVAL', True))
features = [
    {
        'features_dir': '/content/generated_augmented_features',
        'sampling_weight': float(CONFIG.get('POSITIVE_SAMPLING_WEIGHT', 5.0)),
        'penalty_weight': 1.0,
        'truth': True,
        'truncation_strategy': 'truncate_start',
        'type': 'mmap',
    },
    {
        'features_dir': '/content/generated_hard_negative_features',
        'sampling_weight': float(CONFIG.get('HARD_NEG_SAMPLING_WEIGHT', 2.0)),
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
    {
        'features_dir': '/content/negative_datasets/speech',
        'sampling_weight': 10.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
    {
        'features_dir': '/content/negative_datasets/dinner_party',
        'sampling_weight': 10.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
    {
        'features_dir': '/content/negative_datasets/no_speech',
        'sampling_weight': 5.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
]

if USE_DP:
    features.append({
        'features_dir': dp_eval_dir,
        'sampling_weight': 0.0,   # Held-out ambient eval ONLY (fixes 1.0 bug)
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'split',
        'type': 'mmap',
    })

train_cfg = {
    'window_step_ms': 10,
    'train_dir': f"/content/trained_models/{CONFIG['WAKE_WORD_NAME']}",
    'features': features,
    'training_steps': steps,
    'positive_class_weight': [1] * n,
    'negative_class_weight': neg_weights,
    'learning_rates': lrs,
    'batch_size': int(CONFIG.get('BATCH_SIZE', 128)),
    'time_mask_max_size': [5] * n,
    'time_mask_count': [1] * n,
    'freq_mask_max_size': [5] * n,
    'freq_mask_count': [1] * n,
    'eval_step_interval': int(CONFIG.get('EVAL_STEP_INTERVAL', 2000)),
    'clip_duration_ms': int(CONFIG['CLIP_DURATION_MS']),
    'minimization_metric': 'ambient_false_positives_per_hour' if USE_DP else None,
    'target_minimization': float(CONFIG.get('TARGET_FALSE_ACCEPTS_PER_HOUR', 0.25)) if USE_DP else 0.9,
    'maximization_metric': 'average_viable_recall',
}

with open('/content/training_parameters.yaml', 'w') as f:
    yaml.dump(train_cfg, f)

gc.collect()
vm = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f"training_parameters.yaml ready | steps={steps}, lrs={lrs}, neg_w={neg_weights}, dinner_party_eval={USE_DP}")
print(f"System RAM free: {vm.available / 1e9:.2f} GB / {vm.total / 1e9:.2f} GB | Disk free: {disk.free / 1e9:.1f} GB")

In [ ]:
# =============================================================================
# Cell 11: Train MixedNet & Export Quantized Streaming TFLite (Two-Process Safe)
# =============================================================================
# Architecture decoded directly from assets/models/alexa.tflite:
#   first_conv_filters=32, first_conv_kernel_size=5, stride=3
#   pointwise_filters="64,64,64,64", repeat_in_block="1, 1, 1, 1"
#   mixconv_kernel_sizes='[5], [9], [13], [21]', residual_connection="0,0,0,0"
#
# Uses a tiny read-only RaggedMmap wrapper on dinner_party_eval when RAM <= 16.5GB
# (keeps whole unmodified clips without rewriting disk mmaps) and runs training
# vs final streaming TFLite export/eval in two separate processes to avoid OOM.

import json, psutil

with open('/content/config.json') as f:
    CONFIG = json.load(f)

vm_total_gb = psutil.virtual_memory().total / 1e9
eval_mode = CONFIG.get('DINNER_PARTY_EVAL_MODE', 'auto')
use_lite = (eval_mode == 'lite') or (eval_mode == 'auto' and vm_total_gb <= 16.5)
max_frames = int(CONFIG.get('DINNER_PARTY_LITE_MINUTES', 25)) * 60 * 100 if use_lite else 0

runner_code = f'''
import os, runpy, sys
import numpy as np
import mmap_ninja.ragged as rmod

MAX_FRAMES = {max_frames}
_RealRaggedMmap = rmod.RaggedMmap

class ReadOnlyCappedRaggedMmap:
    def __init__(self, out_dir, *args, **kwargs):
        self._mm = _RealRaggedMmap(out_dir, *args, **kwargs)
        self._n = len(self._mm)
        self._last_cap = None
        norm_dir = str(out_dir).rstrip('/')
        if MAX_FRAMES > 0 and 'dinner_party_eval' in norm_dir and not norm_dir.endswith('shapes') and self._n > 0:
            total = 0
            keep = 0
            for i in range(self._n):
                f = int(self._mm[i].shape[0])
                if total + f > MAX_FRAMES:
                    if keep == 0:
                        self._last_cap = MAX_FRAMES
                        keep = 1
                        total = MAX_FRAMES
                    else:
                        rem = MAX_FRAMES - total
                        if rem >= 1800:
                            self._last_cap = rem
                            keep += 1
                            total += rem
                    break
                total += f
                keep += 1
            self._n = max(1, keep)
            print(f'[FreeColab SafeEval] {{os.path.relpath(norm_dir, "/content/negative_datasets")}}: using {{self._n}}/{{len(self._mm)}} clips ({{total / 6000.0:.1f}} min)')

    def __len__(self):
        return self._n

    def _get_item(self, i):
        if i < 0:
            i += self._n
        if i < 0 or i >= self._n:
            raise IndexError(i)
        arr = self._mm[i]
        if self._last_cap is not None and i == self._n - 1 and arr.shape[0] > self._last_cap:
            return np.array(arr[:self._last_cap], copy=True)
        return arr

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return [self._get_item(i) for i in range(*idx.indices(self._n))]
        return self._get_item(int(idx))

    def __iter__(self):
        for i in range(self._n):
            yield self._get_item(i)

    @property
    def shapes(self):
        s = np.array(self._mm.shapes[:self._n], copy=True)
        if self._last_cap is not None and len(s) > 0:
            s[-1][0] = min(int(s[-1][0]), self._last_cap)
        return s

    def __getattr__(self, name):
        return getattr(self._mm, name)

if MAX_FRAMES > 0:
    rmod.RaggedMmap = ReadOnlyCappedRaggedMmap

sys.argv = ['microwakeword.model_train_eval'] + sys.argv[1:]
runpy.run_module('microwakeword.model_train_eval', run_name='__main__')
'''

with open('/content/run_train_eval.py', 'w') as rf:
    rf.write(runner_code)

print(f"Launcher ready (RAM={vm_total_gb:.1f} GB, lite_cap={'OFF (full)' if max_frames == 0 else str(max_frames // 6000) + ' min/split'})")

%cd /content

# Phase A: Train & select best_weights (exits afterward to free all training/optimizer RAM)
!TF_FORCE_GPU_ALLOW_GROWTH=true TF_CPP_MIN_LOG_LEVEL=1 python /content/run_train_eval.py \
--training_config='/content/training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 0 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [9], [13], [21]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

# Phase B: Fresh process with full free RAM -> convert best_weights & evaluate streaming INT8 TFLite
!TF_FORCE_GPU_ALLOW_GROWTH=true TF_CPP_MIN_LOG_LEVEL=1 python /content/run_train_eval.py \
--training_config='/content/training_parameters.yaml' \
--train 0 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [9], [13], [21]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# =============================================================================
# Cell 12: Verify WakeWordEngine.cpp Compatibility, Calibrate Cutoff & Export
# =============================================================================

import glob, json, os, shutil, struct
import numpy as np
import scipy.io.wavfile
from ai_edge_litert.interpreter import Interpreter
from pymicro_features import MicroFrontend

with open('/content/config.json') as f:
    CONFIG = json.load(f)

WW_NAME = CONFIG['WAKE_WORD_NAME']
TRAIN_DIR = f'/content/trained_models/{WW_NAME}'
WINDOW_SIZE = int(CONFIG.get('SLIDING_WINDOW_SIZE', 5))

# 1. Locate quantized streaming .tflite model
candidates = sorted(glob.glob(f'{TRAIN_DIR}/**/*.tflite', recursive=True))
quant_models = [p for p in candidates if 'stream_state_internal_quant' in p]
model_path = quant_models[0] if quant_models else (candidates[0] if candidates else None)
assert model_path and os.path.exists(model_path), f'No .tflite found in {TRAIN_DIR}! Check Cell 11 logs.'
print(f'Found exported model: {model_path} ({os.path.getsize(model_path):,} bytes)')

# Print official microWakeWord evaluation logs/metrics if saved
for txt in sorted(glob.glob(f'{TRAIN_DIR}/**/*.txt', recursive=True)):
    print(f'\n--- {os.path.relpath(txt, TRAIN_DIR)} ---')
    with open(txt) as tf_log:
        print(tf_log.read()[:2500])

# 2. Verify TFLite FlatBuffer operators match Kiwi WakeWordEngine.cpp's 12-op resolver
def inspect_tflite_ops(path):
    with open(path, 'rb') as f:
        buf = f.read()
    def u32(off): return struct.unpack_from('<I', buf, off)[0]
    def i32(off): return struct.unpack_from('<i', buf, off)[0]
    def u16(off): return struct.unpack_from('<H', buf, off)[0]
    def tfield(tpos, fidx):
        vpos = tpos - i32(tpos)
        vsz = u16(vpos)
        off = 4 + fidx * 2
        if off >= vsz:
            return None
        rel = u16(vpos + off)
        return tpos + rel if rel else None
    def vec_offs(pos):
        if pos is None:
            return []
        vp = pos + u32(pos)
        n = u32(vp)
        return [vp + 4 + i * 4 + u32(vp + 4 + i * 4) for i in range(n)]
    BUILTIN = {
        2: 'CONCATENATION', 3: 'CONV_2D', 4: 'DEPTHWISE_CONV_2D',
        9: 'FULLY_CONNECTED', 14: 'LOGISTIC', 22: 'RESHAPE',
        45: 'STRIDED_SLICE', 114: 'QUANTIZE', 129: 'CALL_ONCE',
        142: 'VAR_HANDLE', 143: 'READ_VARIABLE', 144: 'ASSIGN_VARIABLE',
    }
    root = u32(0)
    ops = []
    for oc in vec_offs(tfield(root, 1)):
        f0, f3 = tfield(oc, 0), tfield(oc, 3)
        code = i32(f3) if f3 else (buf[f0] if f0 else 0)
        ops.append(BUILTIN.get(code, f'OP_{code}'))
    return ops

ALLOWED_OPS = {
    'CALL_ONCE', 'VAR_HANDLE', 'RESHAPE', 'READ_VARIABLE',
    'STRIDED_SLICE', 'CONCATENATION', 'ASSIGN_VARIABLE',
    'CONV_2D', 'DEPTHWISE_CONV_2D', 'FULLY_CONNECTED',
    'LOGISTIC', 'QUANTIZE',
}
model_ops = inspect_tflite_ops(model_path)
unsupported = set(model_ops) - ALLOWED_OPS
print(f'Model TFLite ops ({len(model_ops)}): {model_ops}')
assert not unsupported, f'Unsupported ops for WakeWordEngine.cpp: {unsupported}'

interp = Interpreter(model_path=model_path)
interp.allocate_tensors()
in_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]
STRIDE = int(in_det['shape'][1])
print(f'Input : shape={in_det["shape"].tolist()}, dtype={in_det["dtype"].__name__}, quant={in_det["quantization"]}')
print(f'Output: shape={out_det["shape"].tolist()}, dtype={out_det["dtype"].__name__}, quant={out_det["quantization"]}')
assert tuple(in_det['shape']) == (1, 3, 40) and in_det['dtype'] == np.int8
assert tuple(out_det['shape']) == (1, 1) and out_det['dtype'] == np.uint8

# 3. Exact WakeWordEngine.cpp streaming simulation on held-out WAVs AND test mmap spectrograms
def esp32_quantize_features(feats):
    '''Convert pymicro_features/SpectrogramGeneration features into WakeWordEngine.cpp int8 tensor values.'''
    arr = np.asarray(feats)
    if np.issubdtype(arr.dtype, np.floating):
        # pymicro_features.MicroFrontend & SpectrogramGeneration already output float32 scaled by 1/25.6 ([0.0, 26.0])
        # Convert back to TFLM FrontendProcessSamples raw uint16 domain ([0..666]) then apply WakeWordEngine.cpp integer math
        raw_u16 = np.round(arr * 25.6).astype(np.int32)
    else:
        raw_u16 = arr.astype(np.int32)
    return np.clip(((raw_u16 * 256 + 333) // 666) - 128, -128, 127).astype(np.int8)

def extract_features_esp32(pcm_i16):
    # Pad 450ms leading/trailing silence so MicroFrontend settles & streaming finishes word end
    padded = np.concatenate([
        np.zeros(2400, dtype=np.int16),
        pcm_i16,
        np.zeros(7200, dtype=np.int16),
    ])
    raw_bytes = padded.tobytes()
    fe = MicroFrontend()
    frames = []
    byte_idx = 0
    # Feed 160 samples (320 bytes = 10ms) per call matching microwakeword/audio/audio_utils.py
    while byte_idx + 320 <= len(raw_bytes):
        out = fe.process_samples(raw_bytes[byte_idx:byte_idx + 320])
        vals = out['features'] if isinstance(out, dict) else out.features
        sread = out['samples_read'] if isinstance(out, dict) else out.samples_read
        if not sread:
            break
        byte_idx += int(sread) * 2
        if vals and len(vals) == 40:
            frames.append(vals)
    if not frames:
        return np.zeros((0, 40), dtype=np.int8)
    return esp32_quantize_features(np.asarray(frames, dtype=np.float32))

def stream_feats_esp32(feats_i8):
    if len(feats_i8) < STRIDE:
        return 0.0
    local_interp = Interpreter(model_path=model_path)
    local_interp.allocate_tensors()
    in_idx = local_interp.get_input_details()[0]['index']
    out_idx = local_interp.get_output_details()[0]['index']
    window = [0] * WINDOW_SIZE
    w_ptr = 0
    ignore = -3
    max_avg = 0.0
    for i in range(0, len(feats_i8) - STRIDE + 1, STRIDE):
        chunk = feats_i8[i:i + STRIDE][np.newaxis, :, :]
        local_interp.set_tensor(in_idx, chunk)
        local_interp.invoke()
        p_u8 = int(local_interp.get_tensor(out_idx)[0][0])
        window[w_ptr] = p_u8
        w_ptr = (w_ptr + 1) % WINDOW_SIZE
        if ignore < 0:
            ignore += 1
        else:
            avg_p = sum(window) / (255.0 * WINDOW_SIZE)
            if avg_p > max_avg:
                max_avg = avg_p
    return max_avg

def max_streaming_prob_from_pcm(pcm_i16):
    if pcm_i16.ndim > 1:
        pcm_i16 = pcm_i16[:, 0]
    feats = extract_features_esp32(pcm_i16.astype(np.int16))
    return stream_feats_esp32(feats)

def collect_test_pcms(wav_dir, limit=200):
    from microwakeword.audio.clips import Clips
    clips_obj = Clips(wav_dir, '*.wav', max_clip_duration_s=None, remove_silence=False, random_split_seed=10, split_count=0.1)
    pcms = []
    gen = None
    for sname in ('test', 'testing'):
        try:
            gen = clips_obj.audio_generator(split=sname, repeat=1)
            break
        except Exception:
            gen = None
    if gen is not None:
        for item in gen:
            arr = item[0] if isinstance(item, (tuple, list)) else item
            arr = np.asarray(arr)
            peak = float(np.max(np.abs(arr))) if arr.size else 0.0
            if np.issubdtype(arr.dtype, np.floating) and peak <= 1.5:
                arr = np.clip(np.round(arr * 32767.0), -32768, 32767).astype(np.int16)
            else:
                arr = np.clip(np.round(arr), -32768, 32767).astype(np.int16)
            pcms.append(arr)
            if len(pcms) >= limit:
                break
    return pcms

pos_pcms = collect_test_pcms('/content/generated_samples', limit=200)
hn_pcms = collect_test_pcms('/content/hard_negatives', limit=200)
bg_files = [os.path.join('/content/audioset_16k', f) for f in sorted(os.listdir('/content/audioset_16k'))[:60]]
bg_pcms = [scipy.io.wavfile.read(p)[1] for p in bg_files]

# Sanity-check frontend output before scoring
sample_feats = extract_features_esp32(pos_pcms[0])
print(f'Frontend sanity: first pos WAV -> {sample_feats.shape} int8 features (min={sample_feats.min()}, max={sample_feats.max()})')
assert sample_feats.shape[0] > 40, f'Frontend extraction too short: {sample_feats.shape}'

print(f'\nRunning ESP32-exact streaming eval on {len(pos_pcms)} clean positives, {len(hn_pcms)} hard-negatives, {len(bg_pcms)} background clips...')
pos_scores = [max_streaming_prob_from_pcm(p) for p in pos_pcms]
hn_scores = [max_streaming_prob_from_pcm(p) for p in hn_pcms]
bg_scores = [max_streaming_prob_from_pcm(p) for p in bg_pcms]

# Also evaluate augmented noisy/reverberant test spectrograms from RaggedMmap
from mmap_ninja.ragged import RaggedMmap
aug_mm = RaggedMmap('/content/generated_augmented_features/testing/wakeword_mmap')
aug_scores = []
for idx in range(min(200, len(aug_mm))):
    q_i8 = esp32_quantize_features(np.array(aug_mm[idx]))
    aug_scores.append(stream_feats_esp32(q_i8))

print(f'Pos score percentiles (clean)     : p10={np.percentile(pos_scores,10):.3f}, p50={np.percentile(pos_scores,50):.3f}, p90={np.percentile(pos_scores,90):.3f}')
print(f'Pos score percentiles (augmented) : p10={np.percentile(aug_scores,10):.3f}, p50={np.percentile(aug_scores,50):.3f}, p90={np.percentile(aug_scores,90):.3f}')

# Parse official tflite_streaming_roc.txt for lowest 0.000-FAPH cutoff on long ambient dinner_party_eval
roc_path = os.path.join(TRAIN_DIR, 'tflite_stream_state_internal_quant', 'tflite_streaming_roc.txt')
roc_zero_faph_cutoff = None
if os.path.exists(roc_path):
    with open(roc_path) as rf:
        for line in rf:
            if line.startswith('Cutoff ') and 'faph=' in line:
                try:
                    co_val = float(line.split('Cutoff ')[1].split(':')[0])
                    faph_val = float(line.split('faph=')[1].strip())
                    if faph_val == 0.0 and (roc_zero_faph_cutoff is None or co_val < roc_zero_faph_cutoff):
                        roc_zero_faph_cutoff = co_val
                except Exception:
                    pass

cutoffs_to_test = sorted(set([0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.88, 0.90, 0.91, 0.92, 0.94, 0.95] + ([roc_zero_faph_cutoff] if roc_zero_faph_cutoff is not None else [])))
print('\nCutoff | Clean Recall | Aug Recall | Hard-Neg FP | Bg Noise FP')
print('-------+--------------+------------+-------------+------------')
min_safe_cutoff = roc_zero_faph_cutoff if roc_zero_faph_cutoff is not None else 0.75
best_cutoff = roc_zero_faph_cutoff if roc_zero_faph_cutoff is not None else float(CONFIG.get('PROBABILITY_CUTOFF', 0.90))
best_score = -1.0
for co in cutoffs_to_test:
    rec_c = sum(s >= co for s in pos_scores) / max(1, len(pos_scores))
    rec_a = sum(s >= co for s in aug_scores) / max(1, len(aug_scores))
    hn_fp = sum(s >= co for s in hn_scores)
    bg_fp = sum(s >= co for s in bg_scores)
    print(f' {co:.2f}  |    {rec_c*100:5.1f}%    |   {rec_a*100:5.1f}%   |   {hn_fp:3d}/{len(hn_scores):3d}   |   {bg_fp:2d}/{len(bg_scores):2d}')
    if hn_fp == 0 and bg_fp == 0 and co >= min_safe_cutoff:
        combined = 0.5 * (rec_c + rec_a)
        if combined > best_score or (abs(combined - best_score) < 1e-6 and abs(co - 0.90) < abs(best_cutoff - 0.90)):
            best_score = combined
            best_cutoff = co

print(f'\nRecommended probability_cutoff: {best_cutoff:.2f} (Alexa uses 0.90; 0-FAPH ambient floor={roc_zero_faph_cutoff})')

# 4. Export deployment bundle: .tflite, .json (v2 manifest), and _model.h (ESP32 C array)
export_dir = f'/content/export_{WW_NAME}'
shutil.rmtree(export_dir, ignore_errors=True)
os.makedirs(export_dir, exist_ok=True)

out_tflite = os.path.join(export_dir, f'{WW_NAME}.tflite')
shutil.copy2(model_path, out_tflite)

manifest = {
    'type': 'micro',
    'wake_word': CONFIG['WAKE_WORD_DISPLAY'],
    'author': 'Kiwi Custom Trained (Alexa-Parity)',
    'model': f'{WW_NAME}.tflite',
    'trained_languages': ['en'],
    'version': 2,
    'micro': {
        'probability_cutoff': round(float(best_cutoff), 2),
        'sliding_window_size': WINDOW_SIZE,
        'feature_step_size': 10,
        'tensor_arena_size': 22348,
        'minimum_esphome_version': '2024.7.0',
    },
}
out_json = os.path.join(export_dir, f'{WW_NAME}.json')
with open(out_json, 'w') as f:
    json.dump(manifest, f, indent=2)

with open(out_tflite, 'rb') as f:
    raw = f.read()

c_var = f'{WW_NAME}_model'
out_hdr = os.path.join(export_dir, f'{c_var}.h')
with open(out_hdr, 'w') as f:
    f.write(f'// Auto-generated from {WW_NAME}.tflite\n')
    f.write('#pragma once\n#include <stdint.h>\n\n')
    f.write(f'alignas(16) const unsigned char {c_var}[] = {{\n')
    for i in range(0, len(raw), 12):
        chunk = ', '.join(f'0x{b:02x}' for b in raw[i:i + 12])
        comma = ',' if i + 12 < len(raw) else ''
        f.write(f'  {chunk}{comma}\n')
    f.write('};\n')
    f.write(f'const unsigned int {c_var}_len = {len(raw)};\n')

bundle_zip = f'/content/{WW_NAME}_bundle.zip'
shutil.make_archive(f'/content/{WW_NAME}_bundle', 'zip', export_dir)
print(f'\nExported bundle: {bundle_zip}')
print(f'  - {out_tflite} ({len(raw):,} bytes)')
print(f'  - {out_json}')
print(f'  - {out_hdr}')

if CONFIG.get('SAVE_TO_GOOGLE_DRIVE', False):
    from google.colab import drive
    drive.mount('/content/drive')
    drv_dir = f'/content/drive/MyDrive/microWakeWord_{WW_NAME}'
    os.makedirs(drv_dir, exist_ok=True)
    shutil.copy2(bundle_zip, os.path.join(drv_dir, f'{WW_NAME}_bundle.zip'))
    print(f'Copied bundle to Google Drive: {drv_dir}')

In [ ]:
# =============================================================================
# Cell 13: Download Exported Model Bundle (.tflite + .json + .h)
# =============================================================================

import json, os
from google.colab import files

with open('/content/config.json') as f:
    CONFIG = json.load(f)

WW_NAME = CONFIG['WAKE_WORD_NAME']
bundle_zip = f'/content/{WW_NAME}_bundle.zip'
assert os.path.exists(bundle_zip), 'Run Cell 12 first to export the bundle!'
print(f'Downloading {bundle_zip} ({os.path.getsize(bundle_zip):,} bytes)...')
files.download(bundle_zip)

## Deploying `"Hey Kiwi"` into Your Kiwi ESP32-S3 Firmware

Inside the downloaded `hey_kiwi_bundle.zip` you get three files matching your existing Alexa setup:
- `hey_kiwi.tflite` → copy to `assets/models/hey_kiwi.tflite`
- `hey_kiwi.json` → copy to `assets/models/hey_kiwi.json` (contains calibrated `probability_cutoff` and `sliding_window_size: 5`)
- `hey_kiwi_model.h` → copy to `src/hardware/hey_kiwi_model.h` (same format as `src/hardware/alexa_model.h`)

Because this notebook used the exact `alexa.tflite` MixedNet layers (`[5], [9], [13], [21]`), `hey_kiwi_model` works with your existing `MicroMutableOpResolver<12>` and `TENSOR_ARENA_SIZE = 20000` in `src/hardware/WakeWordEngine.cpp` without adding any extra TFLite ops.